In [ ]:
!pip install bitsandbytes
!pip install nltk python-Levenshtein
import nltk
nltk.download('punkt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import torch
import gc
import numpy as np
import bitsandbytes
from PIL import Image
from datasets import load_dataset, concatenate_datasets
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, TrainingArguments, Trainer, DataCollatorForSeq2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein
from tqdm import tqdm


def clear_gpu_memory():
    torch.cuda.empty_cache()
    gc.collect()

def compute_exact_match(pred, ref):
    return pred.strip() == ref.strip()

def compute_bleu(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = [ref.split()]
    return sentence_bleu(ref_tokens, pred_tokens,
                         weights=(0.25, 0.25, 0.25, 0.25),
                         smoothing_function=SmoothingFunction().method1)

def compute_levenshtein_similarity(pred, ref):
    distance = Levenshtein.distance(pred, ref)
    max_len = max(len(pred), len(ref))
    if max_len == 0:
        return 1.0
    return 1 - distance / max_len

def evaluate_predictions(predictions, references):
    em = np.mean([compute_exact_match(p, r) for p, r in zip(predictions, references)])
    bleu = np.mean([compute_bleu(p, r) for p, r in zip(predictions, references)])
    lev = np.mean([compute_levenshtein_similarity(p, r) for p, r in zip(predictions, references)])
    return {"exact_match": em, "bleu": bleu, "levenshtein_sim": lev}

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
latex_train = load_dataset("linxy/LaTeX_OCR", "human_handwrite", split='train')
latex_test = load_dataset("linxy/LaTeX_OCR", "human_handwrite", split='test')
mathwriting = load_dataset("deepcopy/MathWriting-human", split='train', columns=['image', 'latex'])
mathwriting = mathwriting.select(range(2500))

README.md: 0.00B [00:00, ?B/s]

human_handwrite/train-00000-of-00001.par(…):   0%|          | 0.00/16.2M [00:00<?, ?B/s]

human_handwrite/validation-00000-of-0000(…):   0%|          | 0.00/961k [00:00<?, ?B/s]

human_handwrite/test-00000-of-00001.parq(…):   0%|          | 0.00/906k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

...

data/test-00000-of-00001-694f317d8b63419(…):   0%|          | 0.00/44.9M [00:00<?, ?B/s]

data/val-00000-of-00001-184984e66f80ed7a(…):   0%|          | 0.00/81.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/229864 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7644 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/15674 [00:00<?, ? examples/s]

In [ ]:
latex_train = latex_train.rename_column("text", "latex")
latex_test = latex_test.rename_column("text", "latex")

In [ ]:
latex_train, latex_test, mathwriting

(Dataset({
     features: ['image', 'latex'],
     num_rows: 1200
 }),
 Dataset({
     features: ['image', 'latex'],
     num_rows: 70
 }),
 Dataset({
     features: ['image', 'latex'],
     num_rows: 2500
 }))

In [ ]:
def format_example(example):

    image = example["image"]
    latex = example["latex"]

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Convert this handwritten formula into LaTeX"}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": latex}
            ]
        }
    ]

    return {
        "messages": messages,
        "image": image
    }

In [ ]:
# квантизация (ускоряем инференс и экономим память)
model_name = "Qwen/Qwen3-VL-4B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quant_config,
)

processor = AutoProcessor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
print(next(model.parameters()).device)

cuda:0


# Zero-shot Inference

In [ ]:
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

model.eval()

base_prompt = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Convert this handwritten formula into LaTeX"}]}],
    tokenize=False,
    add_generation_prompt=True
)

predictions = []
references = []

for idx, example in enumerate(tqdm(latex_test, desc="Zero-shot inference")):
    image = example["image"]
    ref = example["latex"]

    try:
        inputs = processor(
            images=image,
            text=base_prompt,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                num_beams=1,
                pad_token_id=processor.tokenizer.pad_token_id
            )

        generated = processor.decode(
            output_ids[0][inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )
    except Exception as e:
        print(f"\nОшибка на примере {idx}: {e}")
        generated = ""

    predictions.append(generated)
    references.append(ref)

zero_shot_metrics = evaluate_predictions(predictions, references)
print("\nZero-shot metrics:")
print(zero_shot_metrics)

clear_gpu_memory()

Zero-shot inference: 100%|██████████| 70/70 [07:46<00:00,  6.66s/it]



Zero-shot metrics:
{'exact_match': np.float64(0.0), 'bleu': np.float64(0.4375596763761225), 'levenshtein_sim': np.float64(0.705416931303833)}


## One-shot Inference

In [ ]:
demo_example = latex_train[0]
demo_image = demo_example["image"]
demo_latex = demo_example["latex"]

def build_one_shot_prompt(test_image, demo_image, demo_latex):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},  # демонстрационное изображение
                {"type": "text", "text": "Convert this handwritten formula into LaTeX"}
            ]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": demo_latex}]
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},  # тестовое изображение
                {"type": "text", "text": "Convert this handwritten formula into LaTeX"}
            ]
        }
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt, [demo_image, test_image]
predictions = []
references = []

for example in latex_test:
    test_image = example["image"]
    ref = example["latex"]

    prompt, images = build_one_shot_prompt(test_image, demo_image, demo_latex)

    inputs = processor(
        images=images,
        text=prompt,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id
        )

    generated = processor.decode(
        output_ids[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
    predictions.append(generated)
    references.append(ref)

one_shot_metrics = evaluate_predictions(predictions, references)
print("One-shot metrics:")
print(one_shot_metrics)

clear_gpu_memory()

One-shot metrics:
{'exact_match': np.float64(0.15714285714285714), 'bleu': np.float64(0.6424174061526279), 'levenshtein_sim': np.float64(0.7958525218274343)}


Заметен прирост по всем метрикам

## SFT using QLoRA (only linxy dataset)

In [ ]:
model.train()
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651


In [ ]:
class VLDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):

        texts = [
            self.processor.apply_chat_template(
                item["messages"],
                tokenize=False,
                add_generation_prompt=False
            )
            for item in batch
        ]

        images = [item["image"] for item in batch]

        batch_enc = self.processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        batch_enc["labels"] = batch_enc["input_ids"].clone()

        return batch_enc

In [ ]:
latex_train = latex_train.map(format_example, remove_columns=latex_train.column_names)
latex_test = latex_test.map(format_example, remove_columns=latex_test.column_names)
mathwriting = mathwriting.map(format_example, remove_columns=mathwriting.column_names)

combined_train = concatenate_datasets([latex_train, mathwriting])

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen3vl-latex-ocr",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=latex_train,
    data_collator=VLDataCollator(processor),
)

clear_gpu_memory()

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
10,8.692734
20,5.644751
30,5.383297
40,5.286863
50,5.192451
60,5.227266
70,5.309755
80,5.152352
90,5.207667
100,5.126481


TrainOutput(global_step=150, training_loss=5.496952794392904, metrics={'train_runtime': 1208.5941, 'train_samples_per_second': 0.993, 'train_steps_per_second': 0.124, 'total_flos': 4070187641717760.0, 'train_loss': 5.496952794392904, 'epoch': 1.0})

In [ ]:
model.save_pretrained("./sft_linxy_lora_adapter")
processor.save_pretrained("./sft_linxy_lora_adapter")

['./sft_linxy_lora_adapter/processor_config.json']

In [ ]:
latex_test_raw = load_dataset("linxy/LaTeX_OCR", "human_handwrite", split='test')
latex_test_raw = latex_test_raw.rename_column("text", "latex")

test_images = [ex["image"] for ex in latex_test_raw]
test_refs = [ex["latex"] for ex in latex_test_raw]

base_prompt = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Convert this handwritten formula into LaTeX"}]}],
    tokenize=False,
    add_generation_prompt=True
)

predictions = []

batch_size = 4
for i in tqdm(range(0, len(test_images), batch_size), desc="LoRA SFT inference"):
    batch_images = test_images[i:i+batch_size]

    inputs = processor(
        images=batch_images,
        text=[base_prompt] * len(batch_images),
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            use_cache=True,
        )

    for j, out in enumerate(output_ids):
        generated = processor.decode(
            out[inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )
        predictions.append(generated)

lora_metrics = evaluate_predictions(predictions, test_refs)
print("\nLoRA SFT metrics:")
print(f"  Exact Match: {lora_metrics['exact_match']:.5f}")
print(f"  BLEU-4: {lora_metrics['bleu']:.5f}")
print(f"  Levenshtein similarity: {lora_metrics['levenshtein_sim']:.5f}")

LoRA SFT inference: 100%|██████████| 18/18 [01:06<00:00,  3.72s/it]


LoRA SFT metrics:
  Exact Match: 0.75714
  BLEU-4: 0.95226
  Levenshtein similarity: 0.95991


## SFT Using linxy/LaTeX_OCR:train + deepcopy/MathWriting-human

In [ ]:
from huggingface_hub import upload_folder
model_name = "Qwen/Qwen3-VL-4B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    quantization_config=quant_config,

)

processor = AutoProcessor.from_pretrained(model_name)

model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = TrainingArguments(
    output_dir="./qwen3vl-latex-ocr-combined",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    push_to_hub=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=combined_train,
    data_collator=VLDataCollator(processor),
)

clear_gpu_memory()

trainer.train()

model.save_pretrained("./sft_combined_lora_adapter")
processor.save_pretrained("./sft_combined_lora_adapter")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

trainable params: 11,796,480 || all params: 4,449,612,288 || trainable%: 0.2651


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Step,Training Loss
10,12.986778
20,7.072550
30,5.767653
40,5.760509
50,5.599465
60,5.689657
70,5.538364
80,5.518996
90,5.476089
100,5.593950


Step,Training Loss
10,12.986778
20,7.072550
30,5.767653
40,5.760509
50,5.599465
60,5.689657
70,5.538364
80,5.518996
90,5.476089
100,5.593950


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-69b52a73-1891ef806139a3cf78b853ff;4239ba96-8e74-4298-a1b6-983ff2890316)

Repository Not Found for url: https://huggingface.co/api/models/KaFFa0/qwen3-vl-comb-lora/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

# Validation

In [ ]:
latex_test_raw = load_dataset("linxy/LaTeX_OCR", "human_handwrite", split='test')
latex_test_raw = latex_test_raw.rename_column("text", "latex")

test_images = [ex["image"] for ex in latex_test_raw]
test_refs = [ex["latex"] for ex in latex_test_raw]


quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=quant_config,
    low_cpu_mem_usage=True,
)

model_combined = PeftModel.from_pretrained(base_model, "KaFFa0/qwen3-vl-comb-lora")
model_combined = model_combined.merge_and_unload()
model_combined.eval()

processor_combined = AutoProcessor.from_pretrained("KaFFa0/qwen3-vl-comb-lora")
if processor_combined.tokenizer.pad_token_id is None:
    processor_combined.tokenizer.pad_token_id = processor_combined.tokenizer.eos_token_id
processor_combined.tokenizer.padding_side = 'left'


base_prompt = processor_combined.apply_chat_template(
    [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Convert this handwritten formula into LaTeX"}]}],
    tokenize=False,
    add_generation_prompt=True
)

predictions_combined = []
batch_size = 4

for i in tqdm(range(0, len(test_images), batch_size), desc="Combined SFT inference"):
    batch_images = test_images[i:i+batch_size]

    inputs = processor_combined(
        images=batch_images,
        text=[base_prompt] * len(batch_images),
        padding=True,
        return_tensors="pt"
    ).to(model_combined.device)

    with torch.inference_mode():
        output_ids = model_combined.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=1,
            pad_token_id=processor_combined.tokenizer.pad_token_id,
            use_cache=True,
        )

    for j, out in enumerate(output_ids):
        generated = processor_combined.decode(
            out[inputs.input_ids.shape[1]:],
            skip_special_tokens=True
        )
        predictions_combined.append(generated)

combined_metrics = evaluate_predictions(predictions_combined, test_refs)
print("\nCombined SFT metrics:")
print(f"  Exact Match: {combined_metrics['exact_match']:.5f}")
print(f"  BLEU-4: {combined_metrics['bleu']:.5f}")
print(f"  Levenshtein similarity: {combined_metrics['levenshtein_sim']:.5f}")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Combined SFT inference: 100%|██████████| 18/18 [01:48<00:00,  6.04s/it]


Combined SFT metrics:
  Exact Match: 0.00000
  BLEU-4: 0.52028
  Levenshtein similarity: 0.73171


Метрики упали, возможно, это связано с другим стилем написания кода и различиями в mathwriting датасете

In [ ]:
for i in range(min(5, len(predictions_combined))):
    print(f"\n--- Пример {i+1} ---")
    print(f"Reference (тест): {test_refs[i][:200]}")
    print(f"Prediction (модель): {predictions_combined[i][:200]}")


--- Пример 1 ---
Reference (тест): \sqrt { b ^ { 2 } - 4 a c }
Prediction (модель): $$\sqrt{b^{2} - 4ac}$$

--- Пример 2 ---
Reference (тест): \sqrt { x - y - z + x ^ { 2 } + y ^ { 2 } + z ^ { 2 } }
Prediction (модель): $$\sqrt{x - y - z + x^2 + y^2 + z^2}$$

--- Пример 3 ---
Reference (тест): \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha }
Prediction (модель): $$ \frac { 2 \tan \alpha } { 1 - \tan ^ { 2 } \alpha } $$

--- Пример 4 ---
Reference (тест): \lim _ { x \rightarrow - 1 } \frac { x ^ { 3 } + 1 } { x + 1 }
Prediction (модель): $$\lim_{x \to -1} \frac{x^3 + 1}{x + 1}$$

--- Пример 5 ---
Reference (тест): 1 + \frac { 1 } { 1 ! } + \frac { 1 } { 2 ! } + \frac { 1 } { 3 ! } + \frac { 1 } { 4 ! }
Prediction (модель): $$1 + \frac{1}{1!} + \frac{1}{2!} + \frac{1}{3!} + \frac{1}{4!}$$
